# 📘 Notebook 03: Tokenisation and Normalisation

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module A: Text as Data · Notebook 3 of 3 in this module · (3 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why **tokenisation** is a set of decisions and not a mechanical step
- Split text into **sentences** and **words** with NLTK
- Remove **stop words**, and say when you should not
- Describe **stemming**, write a simple stemmer by hand, and use the Porter stemmer
- Describe **lemmatisation** and explain how it differs from stemming
- Assemble a reusable **preprocessing pipeline** and measure what each step does to the vocabulary

> **Prerequisites:** Notebooks 01 and 02.
>
> 🔭 **Why this notebook matters:** the first thing any NLP system does to a text is cut it into units and decide which of them count as "the same". Every later stage inherits those decisions and cannot undo them. Two systems that disagree here will disagree everywhere, which is why a result you cannot reproduce is so often traced back to preprocessing.

> ℹ️ **Setup note.** This notebook uses NLTK with a few small data packages. Run the cell below once. It takes a few seconds.

In [ ]:
import nltk

for package in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "gutenberg"]:
    nltk.download(package, quiet=True)

import re
from collections import Counter

## 1. What is a word, exactly?

### Intuition first
In Notebook 01 a word was "whatever lies between two spaces". In Notebook 02 it was "a run of letters". Both definitions are convenient, and both break as soon as the text becomes interesting. Consider this sentence:

*"Dr. O'Neill didn't pay $3.50 for the state-of-the-art e-reader in New York."*

Is `didn't` one word or two (*did* + *not*)? Is `$3.50` one unit, or a currency sign and a number? Is `New York` two words or one name? Is the full stop after `Dr` part of the abbreviation or the end of a sentence? There is no single correct answer. There are only **choices**, and each tokeniser makes them differently.

### Formal definition
**Tokenisation** is the task of segmenting a text into units called **tokens**. Tokens are usually words, numbers and punctuation marks, but as we will see in Notebook 15 they can also be pieces of words.

Let us put three tokenisers side by side.

In [ ]:
from nltk.tokenize import word_tokenize

sentence = "Dr. O'Neill didn't pay $3.50 for the state-of-the-art e-reader in New York."

by_spaces = sentence.split()
by_pattern = re.findall(r"[a-z]+(?:'[a-z]+)*", sentence.lower())
by_nltk = word_tokenize(sentence)

print("spaces: ", by_spaces)
print("pattern:", by_pattern)
print("NLTK:   ", by_nltk)

Compare them carefully.

- **Splitting on spaces** leaves punctuation glued on: `York.` is not `York`.
- **Our pattern** from Notebook 02 lost the price entirely and broke `e-reader` into `e` and `reader`.
- **NLTK** keeps punctuation as separate tokens, keeps `$` and `3.50` as two meaningful units, keeps `Dr.` together, and splits `didn't` into `did` and `n't`.

That last choice surprises people. It follows a long-standing convention for English: the negation is a separate unit of meaning, so it becomes a separate token. A system that needs to notice negation will be glad of it.

> 🧠 None of the three is "right". A tokeniser is right **for a task**. What matters is that you know which choices yours makes, and that you use the **same** tokeniser when you train a system and when you use it.

## 2. Splitting text into sentences

### Intuition first
Many tasks work sentence by sentence: translation, summarisation, parsing. The obvious rule is to cut at every full stop. Exercise 6 of Notebook 02 showed where that leads.

In [ ]:
from nltk.tokenize import sent_tokenize

paragraph = "Dr. Smith arrived at 9 a.m. on Monday. Was he late? Not at all! He paid 3.50 euros for coffee."

print("Cutting at every full stop:")
for piece in paragraph.split(". "):
    print("  ", repr(piece))

print("\nNLTK sentence tokeniser:")
for sentence in sent_tokenize(paragraph):
    print("  ", repr(sentence))

The simple rule cut after `Dr.` and after `a.m.`, and it ignored `?` and `!` altogether. NLTK's tokeniser got all four sentences right.

It can do this because it was **trained**: it was shown a large amount of English text and learned which words are usually abbreviations and which words usually begin sentences. It is our first example of a pattern that will repeat throughout the course. Rules written by hand are replaced by behaviour learned from data.

> ⚠️ **Common pitfalls**
>
> - Assuming a sentence tokeniser is perfect. Feed it text unlike what it was trained on (chat messages, legal clauses, text without capital letters) and it will make mistakes.
> - Using an English tokeniser for another language. The abbreviations and conventions differ. NLTK accepts a `language` argument for this reason.

## 3. Stop words

### Intuition first
In Notebook 01 the most frequent words of *Alice in Wonderland* were *the*, *and*, *to*, *a*. They appear in every English text and tell us nothing about this one. Words like these are called **stop words**. If we want to know what a text is about, a natural idea is to remove them.

### Formal definition
A **stop-word list** is simply a fixed list of very common words to be discarded. NLTK provides one for English and for many other languages.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print(len(stop_words), "English stop words")
print(sorted(stop_words)[:30])

In [ ]:
from nltk.corpus import gutenberg

alice = gutenberg.raw("carroll-alice.txt")
tokens = re.findall(r"[a-z]+(?:'[a-z]+)*", alice.lower())

content = []
for token in tokens:
    if token not in stop_words:
        content.append(token)

print(f"All tokens:         {len(tokens):,}")
print(f"Without stop words: {len(content):,}")
print(f"Removed:            {1 - len(content) / len(tokens):.0%}")
print()
print("Before:", [word for word, count in Counter(tokens).most_common(12)])
print("After: ", [word for word, count in Counter(content).most_common(12)])

More than half of the tokens were stop words, exactly as Zipf's law predicted. What is left starts to look like the book: *alice*, *little*, *thought*, *queen*. It also shows that no list is perfect: *said*, *one* and *would* tell us little, and they are still here.

A stop-word list is also the first thing in this course that is specific to **one language**. NLTK has lists for others:

In [ ]:
print(stopwords.words("greek")[:15])
print(stopwords.words("french")[:15])

### When removing stop words is a mistake
Look again at the English list. It contains *not*, *no*, *don't* and *won't*.

In [ ]:
def remove_stop_words(text):
    kept = []
    for token in re.findall(r"[a-z]+(?:'[a-z]+)*", text.lower()):
        if token not in stop_words:
            kept.append(token)
    return kept

print(remove_stop_words("This film is good."))
print(remove_stop_words("This film is not good."))

After stop-word removal, a positive review and a negative review are **identical**. For a task that depends on negation, such as judging opinions, the standard stop-word list deletes the most important word in the sentence.

> ⚠️ **Common pitfalls**
>
> - Removing stop words out of habit. It helps when you want the **topic** of a text. It hurts when the small words carry the meaning: opinions (*not good*), questions (*who*, *where*), names and titles (*The Who*, *To be or not to be*).
> - Treating the list as fixed. You may add words that are useless in your own domain (*said* in a novel, *patient* in medical notes) and take out words you need (*not*).

## 4. Stemming

### Intuition first
`connect`, `connects`, `connected`, `connecting` and `connection` are five different strings. For a search engine they should all lead to the same documents. **Stemming** is the crude and fast answer: chop the ending off a word and hope that related words end up identical.

### A stemmer by hand
Here is the whole idea in a few lines: a list of endings, tried from longest to shortest.

In [ ]:
def simple_stem(word):
    for ending in ["ations", "ation", "ings", "ing", "ies", "ed", "es", "ly", "s"]:
        # only cut if a reasonable amount of the word is left
        if word.endswith(ending) and len(word) - len(ending) >= 3:
            return word[:-len(ending)]
    return word

for word in ["connects", "connected", "connecting", "walking", "cats", "quickly", "is", "this"]:
    print(f"{word:<11} -> {simple_stem(word)}")

It works on the easy cases. Now the hard ones:

In [ ]:
for word in ["running", "studies", "wolves", "king", "news", "caring"]:
    print(f"{word:<9} -> {simple_stem(word)}")

`running` became `runn`, `studies` became `stud`, and two results are different words altogether: `caring` became `car` and `news` became `new`. Only `king` survived, saved by the rule about minimum length. English spelling has too many special cases for nine rules.

### Formal definition
A **stemmer** applies an ordered set of suffix-rewriting rules. The result, the **stem**, need not be a real word. The best known is the **Porter stemmer** (1980), with about sixty rules arranged in stages, and conditions on what must remain of the word before a rule may fire.

In [ ]:
from nltk.stem import PorterStemmer

porter = PorterStemmer()

print(f"{'word':<13} {'ours':<11} {'Porter'}")
for word in ["connected", "connection", "running", "studies", "king", "news", "caring", "happily", "relational"]:
    print(f"{word:<13} {simple_stem(word):<11} {porter.stem(word)}")

Porter handles the doubled consonant in `running`, leaves `news` alone, turns `caring` into `care`, and maps `connected` and `connection` to the same stem. It also produces `studi`, `happili` and `relat`, which are not words. That is by design: a stem is a **key for matching**, not something to show to a reader.

A stemmer can fail in two directions.

In [ ]:
print("Over-stemming (different words, same stem):")
for word in ["university", "universe", "universal"]:
    print("  ", word, "->", porter.stem(word))

print("Under-stemming (same word family, different stems):")
for word in ["alumnus", "alumni", "ran", "run"]:
    print("  ", word, "->", porter.stem(word))

A *university* and the *universe* have little in common, yet they share a stem. Meanwhile `ran` and `run` are forms of one verb and stay apart, because no suffix rule can connect them.

## 5. Lemmatisation

### Intuition first
A stemmer cuts. A **lemmatiser** looks the word up. It returns the form you would find as the headword in a dictionary: `ran` becomes `run`, `mice` becomes `mouse`, `better` becomes `good`. This dictionary form is called the **lemma**.

### Formal definition
**Lemmatisation** maps each word to its lemma, using a vocabulary and knowledge of the word's grammatical role. NLTK's lemmatiser relies on **WordNet**, a large hand-built dictionary of English.

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatiser = WordNetLemmatizer()

print(f"{'word':<10} {'stem':<10} {'lemma'}")
for word in ["mice", "geese", "studies", "wolves", "churches", "universities"]:
    print(f"{word:<10} {porter.stem(word):<10} {lemmatiser.lemmatize(word)}")

Every lemma is a real word, and the irregular plurals are handled correctly. There is a catch, though.

In [ ]:
for word in ["running", "ran", "better", "was"]:
    print(f"{word:<8} as a noun: {lemmatiser.lemmatize(word):<8} as a verb: {lemmatiser.lemmatize(word, pos='v'):<6} as an adjective: {lemmatiser.lemmatize(word, pos='a')}")

By default the lemmatiser assumes every word is a **noun**. Then `running` and `ran` are left unchanged, and `was` even becomes `wa`, as if it were a plural. Only when we tell it the word is a verb does it answer `run` and `be`, and only as an adjective does `better` become `good`.

So a lemmatiser needs to know the **part of speech** of each word, and the same string can have different lemmas: *leaves* belongs to *leaf* when it is a noun, and to *leave* when it is a verb. Working out the part of speech automatically is the subject of Notebook 06.

### Stemming or lemmatisation?

| | Stemming | Lemmatisation |
|---|---|---|
| Method | cut endings by rule | look up in a dictionary |
| Result | a stem, often not a word | a real word |
| Speed | very fast | slower |
| Needs | nothing | a dictionary and the part of speech |
| Typical mistakes | over- and under-stemming | wrong answer when the part of speech is wrong |
| Good for | search, quick experiments | when the output must be readable or accurate |

## 6. Putting it together: a preprocessing pipeline

Each technique in this module is a step that takes tokens in and gives tokens out. Chained together they form a **preprocessing pipeline**. Here is one with switches, so that every step can be turned on or off.

In [ ]:
def preprocess(text, lower=True, remove_stops=False, stem=False):
    if lower:
        text = text.lower()
    tokens = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)*", text)

    result = []
    for token in tokens:
        if remove_stops and token.lower() in stop_words:
            continue
        if stem:
            token = porter.stem(token)
        result.append(token)
    return result

sample = "The children were running quickly towards the old churches."

print(preprocess(sample, lower=False))
print(preprocess(sample))
print(preprocess(sample, remove_stops=True))
print(preprocess(sample, remove_stops=True, stem=True))

Now the measurement. How much does each step shrink the vocabulary of a whole book?

In [ ]:
settings = [
    ("raw tokens",        dict(lower=False)),
    ("+ lower case",      dict()),
    ("+ stop words out",  dict(remove_stops=True)),
    ("+ stemming",        dict(remove_stops=True, stem=True)),
]

print(f"{'pipeline':<20} {'tokens':>8} {'types':>7}")
for name, options in settings:
    result = preprocess(alice, **options)
    print(f"{name:<20} {len(result):>8,} {len(set(result)):>7,}")

Read the table from top to bottom. Lower-casing merges `The` with `the` and removes a few hundred types. Removing stop words barely changes the number of **types**, since there are only a couple of hundred of them, but it more than halves the number of **tokens**. Stemming then cuts the vocabulary by roughly a quarter, by merging the different forms of each word.

A smaller vocabulary means fewer things for a model to learn, and more examples of each. That was essential for the counting methods of Module C, which is where you will use this pipeline.

> 🧠 **A note for later.** The large neural language models at the end of this course do almost **none** of this. They keep capital letters, punctuation and stop words, and they never stem. They have enough data to learn for themselves that *running* and *ran* are related, and enough capacity to make use of the difference between *"Good."* and *"good?"*. Heavy preprocessing is a tool for small data and simple models. Knowing when to leave text alone is part of the skill.

---
## ✏️ Exercises

### Exercise 1 (Compare tokenisers)
Tokenise the sentence below in three ways: `split()`, the pattern `[a-z]+(?:'[a-z]+)*` on the lower-cased text, and `word_tokenize`. Print the number of tokens each gives. Which treats `U.K.` and `can't` most sensibly?

In [ ]:
text = "I can't believe the U.K. team won 3-2... again!"
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
text = "I can't believe the U.K. team won 3-2... again!"

for name, result in [
    ("split", text.split()),
    ("pattern", re.findall(r"[a-z]+(?:'[a-z]+)*", text.lower())),
    ("nltk", word_tokenize(text)),
]:
    print(f"{name:<8} {len(result):>2}  {result}")
```

*The pattern breaks "U.K." into "u" and "k" and loses the score. NLTK keeps "U.K." whole, splits "can't" into "ca" and "n't", and keeps the score and the dots as tokens.*
</details>

### Exercise 2 (Sentences in a book)
Split *Alice in Wonderland* into sentences with `sent_tokenize`. How many are there? What is the average sentence length in words, and what is the longest sentence?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
sentences = sent_tokenize(alice)
lengths = []
for sentence in sentences:
    lengths.append(len(sentence.split()))

print(len(sentences), "sentences")
print(f"Average length: {sum(lengths) / len(lengths):.1f} words")

longest = sentences[lengths.index(max(lengths))]
print(f"Longest: {max(lengths)} words")
print(longest[:300], "...")
```

*Nineteenth-century prose has very long sentences. Any system with a limit on input length has to plan for the long tail, not for the average.*
</details>

### Exercise 3 (Your own stop-word list)
In *Alice*, the word `said` is extremely common and says little. Create a custom stop-word set that adds `said` to the standard list and **removes** `not` and `no` from it. Print the 10 most common content words of the book with your list.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
my_stop_words = set(stop_words)
my_stop_words.add("said")
my_stop_words.discard("not")
my_stop_words.discard("no")

kept = []
for token in tokens:
    if token not in my_stop_words:
        kept.append(token)

print(Counter(kept).most_common(10))
```

*We copied the set before changing it, so that the original list stays intact. A stop-word list is a setting you tune, not a fact about the language.*
</details>

### Exercise 4 (How good is the stemmer?)
For each group below, apply the Porter stemmer and decide: are the words that belong together given the same stem? Are any words wrongly merged?

In [ ]:
groups = [
    ["organise", "organised", "organisation", "organ"],
    ["policy", "police", "policies"],
    ["general", "generate", "generous"],
]
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
groups = [
    ["organise", "organised", "organisation", "organ"],
    ["policy", "police", "policies"],
    ["general", "generate", "generous"],
]

for group in groups:
    for word in group:
        print(f"{word:<13} -> {porter.stem(word)}")
    print()
```

*The first two groups work well: the forms of "organise" share a stem while "organ" is kept apart, and "policy" and "policies" are merged while "police" stays separate. The third group is a classic case of over-stemming: "general", "generate" and "generous" have nothing to do with each other, and all three collapse into "gener". A stemmer is a bundle of compromises. Always look at what it does to the words that matter in your data.*
</details>

### Exercise 5 (Lemmas need grammar)
Lemmatise each word of the list `["leaves", "saw", "meeting", "left"]` twice: once as a noun (`pos="n"`) and once as a verb (`pos="v"`). For which words do the two answers differ, and why?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for word in ["leaves", "saw", "meeting", "left"]:
    as_noun = lemmatiser.lemmatize(word, pos="n")
    as_verb = lemmatiser.lemmatize(word, pos="v")
    print(f"{word:<9} noun: {as_noun:<9} verb: {as_verb}")
```

*Each of these strings is two different words. "leaves" is the plural of "leaf" or a form of "leave", and "left" is a direction or the past of "leave". The string alone does not tell you which. Only the sentence around it does. Notice also that "saw" stays "saw" even as a verb: "to saw" is a verb in its own right, so the lemmatiser has no way of knowing that you meant the past of "see".*
</details>

### Exercise 6 (Stems or lemmas, measured)
Using the tokens of *Alice* with stop words removed (the list `content` from Section 3), compute the vocabulary size three ways: unchanged, after Porter stemming, and after lemmatisation (try the verb form first, then the noun form). Are the two reduced vocabularies similar in size? Are they similar in content?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
stems = set()
lemmas = set()
for token in content:
    stems.add(porter.stem(token))
    lemmas.add(lemmatiser.lemmatize(lemmatiser.lemmatize(token, pos="v"), pos="n"))

print(f"Unchanged:  {len(set(content)):,} types")
print(f"Stemmed:    {len(stems):,} types")
print(f"Lemmatised: {len(lemmas):,} types")
print(sorted(stems)[:12])
print(sorted(lemmas)[:12])
```

*The two sizes are almost the same, but the contents are not. The stems include strings that are not words, and some of the merges are wrong. Every lemma is a real word, and the forms it merges really are the same word. Counting types tells you how much was merged. It does not tell you whether the merging was correct.*
</details>

### Exercise 7 (What is this book about? A little harder)
Write a function `keywords(text, n)` that returns the `n` most frequent **lemmas** of a text after removing stop words and words shorter than 3 letters. Run it on *Alice* and on `gutenberg.raw("melville-moby_dick.txt")`. Could you guess each book from its ten keywords?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def keywords(text, n):
    kept = []
    for token in re.findall(r"[a-z]+", text.lower()):
        if token in stop_words or len(token) < 3:
            continue
        kept.append(lemmatiser.lemmatize(token))
    return Counter(kept).most_common(n)

print(keywords(alice, 10))
print(keywords(gutenberg.raw("melville-moby_dick.txt"), 10))
```

*Moby Dick is unmistakable: whale, ship, sea, ahab, boat. Alice is harder. Once the name is set aside, its most frequent words are everyday ones such as "little", "know" and "like". Raw frequency favours words that are common everywhere. Notebook 08 repairs this by rewarding words that are frequent in one text and rare in the others.*
</details>

## 🔑 Key takeaways

- **Tokenisation** is a series of choices about contractions, punctuation, numbers and names. Know the choices your tokeniser makes, and keep it the same throughout a project.
- **Sentence splitting** is harder than cutting at full stops. Trained tokenisers learn abbreviations from data.
- **Stop words** are frequent words with little content. Removing them reveals the topic and can destroy meaning, above all negation.
- **Stemming** chops endings by rule. It is fast and crude, and produces stems that need not be words.
- **Lemmatisation** returns the dictionary form. It is more accurate and needs the part of speech.
- Each preprocessing step **shrinks the vocabulary** and discards information. Measure the effect and choose for the task.
- Small data and simple models benefit from heavy preprocessing. Large neural models prefer the text left almost untouched.

### 🏁 You have completed Module A: Text as Data!
You can now take raw text in any state, clean it, cut it into tokens and reduce it to a manageable vocabulary. So far every word has been treated as an isolated item. The next module looks at how words combine: the rules that make *"the dog bit the man"* a sentence and *"dog the the bit man"* a heap of words.

**Next:** *Module B: The Structure of Language*, starting with *Notebook 04: Formal Languages and Automata*.

---
*End of Module A.*

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*